# DSP501 — Analysis Dashboard

**Noise-robust speaker embedding with a designed DSP front-end.**

This notebook is the *analysis* layer. It reads artifacts written by the CLI
and never re-runs the study, so the numbers here cannot drift from the ones in
the report. Produce the artifacts first:

```bash
uv run python run.py all --config configs/dsp501-v2.json
```

| Layer | File | Role |
|---|---|---|
| Execution | `run.py` | Canonical, reproducible study run |
| Analysis | `main.ipynb` | This notebook: inspect and explore saved artifacts |
| Delivery | `report.qmd` | Final report rendered from the same artifacts |

In [ ]:
import csv
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.config import load_config

CONFIG_PATH = Path("configs/dsp501-v2.json")
config = load_config(CONFIG_PATH)
REPORTS = config.report_root

print(f"study      : {config.study_id}")
print(f"seeds      : {config.seeds}")
print(f"test SNRs  : {config.test_snr_db} dB")
print(f"conditions : {[c.name for c in config.conditions]}")
print(f"artifacts  : {REPORTS}")

## 1. DSP front-end, stage by stage

One deterministic held-out clip, mixed at the hardest test SNR, then passed
through each stage of Pipeline B. Listen to the audio and watch the residual
noise floor drop in the spectrogram.

In [ ]:
from IPython.display import Audio, display
from scipy import signal as scipy_signal

from src import dsp
from src.corpus import load_clips, load_manifest
from src.noise import load_noise_pool_for_split, mix_at_snr, noise_for_sample
from src.pipeline import build_chain

CLIP_INDEX = 0
DEMO_SNR_DB = float(config.test_snr_db[0])

records = load_manifest(config.cache_root, "test")
clips = load_clips(config.cache_root, "test")
pool = load_noise_pool_for_split(config.cache_root, "test")
record = records[CLIP_INDEX]

clean = clips[CLIP_INDEX]
noisy = mix_at_snr(clean, noise_for_sample(record.sample_id, config.seeds[0], pool), DEMO_SNR_DB)

stage_names = config.condition("B_full").stages
all_stages = build_chain(config.condition("B_full"), config).stages
signals = {"clean": clean, f"noisy @ {DEMO_SNR_DB:.0f} dB": noisy}
for depth in range(1, len(stage_names) + 1):
    partial = dsp.DspChain(names=stage_names[:depth], stages=all_stages[:depth])
    signals["+".join(stage_names[:depth])] = partial(noisy)

print(f"clip {record.sample_id} (speaker {record.speaker_id})")
for name, waveform in signals.items():
    print(f"  {name}")
    display(Audio(waveform, rate=config.sample_rate))

In [ ]:
figure, axes = plt.subplots(len(signals), 1, figsize=(11, 2.0 * len(signals)), sharex=True)
for axis, (name, waveform) in zip(axes, signals.items()):
    freqs, times, spectrum = scipy_signal.stft(
        waveform,
        fs=config.sample_rate,
        nperseg=config.n_fft,
        noverlap=config.n_fft - config.hop_length,
    )
    axis.pcolormesh(
        times,
        freqs,
        20 * np.log10(np.abs(spectrum) + 1e-8),
        shading="auto",
        vmin=-100,
        vmax=-20,
        cmap="magma",
    )
    axis.set_ylabel(name, fontsize=7, rotation=0, ha="right", va="center")
axes[-1].set_xlabel("time (s)")
figure.suptitle("STFT magnitude through the Pipeline B stages")
figure.tight_layout()

## 2. Data-driven filter design

Cut-offs come from the measured spectra, not from convention. A band is worth
removing only when the noise there dominates the speech.

In [ ]:
bands = json.loads((REPORTS / "band-analysis.json").read_text(encoding="utf-8"))

print(f"{'band (Hz)':>16} {'speech %':>9} {'noise %':>9} {'band SNR dB':>12}")
for band in bands["bands"]:
    print(
        f"{int(band['low_hz']):>7}-{int(band['high_hz']):<8}"
        f" {band['speech_power_pct']:>9.2f} {band['noise_power_pct']:>9.2f}"
        f" {band['band_snr_db']:>12.2f}"
    )
print()
print(json.dumps(bands["recommended_cutoffs"], indent=2))

In [ ]:
from IPython.display import Image

for name in ("band-analysis.png", "filter-response.png"):
    display(Image(filename=str(REPORTS / name)))

## 3. Results

Means over seeds and test SNRs, on speaker-disjoint held-out speakers.

In [ ]:
with (REPORTS / "metric-summary.csv").open(newline="", encoding="utf-8") as stream:
    metrics = list(csv.DictReader(stream))


def mean_of(condition, metric, model="cnn"):
    values = [
        float(row[metric])
        for row in metrics
        if row["condition"] == condition and row["model"] == model and row[metric]
    ]
    return sum(values) / len(values) if values else float("nan")


columns = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "agglomerative_ari", "agglomerative_nmi"]
header = f"{'arm':<20}" + "".join(f"{c.replace('_macro', '').replace('agglomerative_', ''):>12}" for c in columns)
print(header)
print("-" * len(header))
for condition in [c.name for c in config.conditions]:
    row = mean_of(condition, "accuracy")
    if row != row:  # arm not evaluated
        continue
    print(f"{condition:<20}" + "".join(f"{mean_of(condition, c):>12.4f}" for c in columns))

In [ ]:
for name in ("snr-curves.png", "ablation-f1.png", "confusion-matrices.png", "embedding-projection.png"):
    path = REPORTS / name
    if path.is_file():
        display(Image(filename=str(path)))

## 4. Statistical significance

Pipeline B minus Pipeline A, paired on seed and test SNR.

In [ ]:
for test in json.loads((REPORTS / "significance.json").read_text(encoding="utf-8")):
    print(f"{test['metric']}  ({test['comparison']}, n={test['n_pairs']})")
    print(f"  mean difference {test['mean_difference']:+.4f}")
    print(f"  95% CI          [{test['ci_low']:+.4f}, {test['ci_high']:+.4f}]")
    print(f"  paired t-test   p = {test['t_p_value']:.4g}")
    print(f"  Wilcoxon        p = {test['wilcoxon_p_value']:.4g}\n")

## 5. Error analysis

Which speakers are confused, and how confident were the failures?

In [ ]:
seed = config.seeds[0]
tag = f"snr-{int(config.test_snr_db[0])}"

for condition in ("A_raw_noisy", "B_full"):
    path = config.run_root(seed, condition) / f"evaluation-{tag}.json"
    if not path.is_file():
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(f"=== {condition} @ {tag} — accuracy {payload['nearest_centroid']['accuracy']:.4f} ===")
    print(f"{len(payload['errors'])} recorded failures; most confused pairs:")
    for pair in payload["confusable_speakers"][:6]:
        print(f"  {pair['true_speaker']} -> {pair['predicted_speaker']}: {pair['count']}")
    print("  highest-confidence errors:")
    for error in payload["errors"][:5]:
        print(
            f"    {error['sample_id']:<32} {error['true_speaker']} -> "
            f"{error['predicted_speaker']}  margin {error['margin']:+.3f}"
        )
    print()

## 6. Training curves

Validation accuracy is measured on *unseen* validation speakers by cosine
nearest-centroid identification, so it tracks embedding generalisation rather
than classifier fit.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
for history_path in sorted(config.study_root.glob(f"seed-{seed}/*/training-history.json")):
    payload = json.loads(history_path.read_text(encoding="utf-8"))
    history = payload["history"]
    label = payload["result"]["condition"]
    axes[0].plot([h["epoch"] for h in history], [h["train_loss"] for h in history], label=label)
    axes[1].plot(
        [h["epoch"] for h in history], [h["validation_accuracy"] for h in history], label=label
    )
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("ArcFace training loss")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("validation accuracy (unseen speakers)")
axes[1].legend(fontsize=7)
for axis in axes:
    axis.grid(alpha=0.3)
figure.tight_layout()